### Setup

Install dependencies

In [ ]:
# # PyTorch
%pip install torch torchvision torchaudio -q

# # HuggingFace ecosystem
%pip install transformers[torch] datasets -q

# # Utilities
%pip install ipywidgets tqdm pyarrow scikit-learn tensorboardX -q

Set huggingface access-token for loading pre-trained models

In [2]:
import os
from huggingface_hub import login
from dotenv import load_dotenv

token = None # "YOUR_TOKEN"

load_dotenv()

hf_token = token or os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token)

Supress unwanted warnings

In [3]:
import warnings

warnings.filterwarnings("ignore")

### Initialize fine-tuning

Import packages

In [4]:
import notebook_init  # Set visible GPUs
import json
import numpy as np
import os
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import ParameterGrid, StratifiedKFold
from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DefaultDataCollator,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

LABEL_ID = {
    "HUMAN_GENERATED": 0,
    "MACHINE_GENERATED": 1,
}

LABEL_TEXT = {
    0: "HUMAN_GENERATED",
    1: "MACHINE_GENERATED"
}

[env_init] Using GPU: 0


Tokenize and chunk

In [ ]:
def chunk_code(tokenizer, code, window=512, stride=0):
    """
    Chunk a single code string into fixed-length token windows.
    Ensures that the FULL code snippet is captured and all chunks are exactly 'window' length.
    """
    # Tokenize the full code without truncation
    full_encoded = tokenizer(
        code,
        add_special_tokens=True,
        truncation=False,
    )
    
    input_ids = full_encoded["input_ids"]
    attention_mask = full_encoded["attention_mask"]
    
    # Split into chunks of size 'window'
    chunks = []
    
    # If stride is 0, we just do a simple split
    step = window if stride <= 0 else (window - stride)
    
    for i in range(0, len(input_ids), step):
        chunk_ids = input_ids[i : i + window]
        chunk_mask = attention_mask[i : i + window]
        
        # Handle the final chunk padding
        if len(chunk_ids) < window:
            padding_length = window - len(chunk_ids)
            # Use tokenizer.pad_token_id explicitly
            chunk_ids = chunk_ids + [tokenizer.pad_token_id] * padding_length
            chunk_mask = chunk_mask + [0] * padding_length
            
        chunks.append({
            "input_ids": chunk_ids,
            "attention_mask": chunk_mask
        })
        
        # Break if we've reached the end of the ids
        if i + window >= len(input_ids):
            break

    return chunks


def map_chunks(batch, tokenizer, window, model_type="encoder"):
    """Map a batch of code/label pairs into tokenized chunk features."""
    all_input_ids, all_attention_mask, all_labels, all_raw_code = [], [], [], []

    for code, label in zip(batch["code"], batch["label"]):
        # Normalize label
        if isinstance(label, str):
            label_key = label.strip().upper()
            if label_key not in LABEL_ID:
                raise ValueError(f"Unknown label: {label}")
            label_id = LABEL_ID[label_key]
        else:
            label_id = int(label)

        if model_type == "causal":
            encoded_chunks = chunk_code(tokenizer, code, window=window, stride=128)
        else:
            encoded_chunks = chunk_code(tokenizer, code, window=window)
        for enc in encoded_chunks:
            all_input_ids.append(enc["input_ids"])
            all_attention_mask.append(enc["attention_mask"])
            all_labels.append(label_id)
            all_raw_code.append(code)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels,
        "raw_code": all_raw_code,
    }


def tokenize_and_chunk(dataset, tokenizer, window, model_type, num_proc=1):
    """Tokenize and chunk an entire dataset into model-ready features."""
    return dataset.map(
        map_chunks,
        batched=True,
        remove_columns=dataset.column_names,
        fn_kwargs={"tokenizer": tokenizer, "window": window, "model_type": model_type},
        num_proc=num_proc
    )


def format_labels(dataset, tokenizer, model_type, window):
    """Format labels for encoder/seq2seq/causal models."""
    if model_type == "seq2seq":
        def format_t5(example):
            return {
                "labels": tokenizer(
                    LABEL_TEXT[example["labels"]],
                    truncation=True
                )["input_ids"]
            }
        return dataset.map(format_t5)

    if model_type == "causal":
        def format_deepseek(sample):
            prompt = (
                "Classify the following code (output either HUMAN_GENERATED or MACHINE_GENERATED):\n\n"
                + sample["raw_code"]
            )
            full_text = prompt + "\n" + LABEL_TEXT[sample["labels"]]
            tok = tokenizer(full_text, truncation=True, max_length=window)
            labels = tok["input_ids"].copy()
            prompt_len = len(tokenizer(prompt, truncation=True, max_length=window)["input_ids"])
            labels[:prompt_len] = [-100] * prompt_len
            tok["labels"] = labels
            return tok
        return dataset.map(format_deepseek)

    return dataset


def load_tokenizer(model_name):
    """Load a tokenizer and ensure a usable pad token is set and configured."""
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    
    # Handle specific model padding requirements
    if "unixcoder" in model_name.lower():
        # UniXcoder uses <pad> but sometimes needs explicit setting
        if tokenizer.pad_token is None:
             tokenizer.add_special_tokens({'pad_token': '<pad>'})
    elif "deepseek" in model_name.lower():
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
    elif tokenizer.pad_token_id is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({'pad_token': '[PAD]'})
            
    return tokenizer


def load_model(model_name, tokenizer):
    """Load appropriate model architecture based on name."""
    model_name_lower = model_name.lower()
    
    if "codet5" in model_name_lower:
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True)
    elif "deepseek" in model_name_lower:
        model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
        model.config.use_cache = False
    else:
        # CodeBERT, GraphCodeBERT, UniXcoder
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2,
            trust_remote_code=True
        )
        # Ensure model config uses the tokenizer pad token
        model.config.pad_token_id = tokenizer.pad_token_id

    return model

Set dataset and config paths

In [6]:
current_dir = Path.cwd()

if current_dir.name == "notebooks" and (
    Path.exists(current_dir.parent / Path("configs/fine_tune"))
    and Path.exists(current_dir.parent / Path("data"))
):
    os.chdir(current_dir.parent)

print(f"Current directory: {Path.cwd().relative_to(Path.home())}")

cfg_path = "configs/fine_tune/codet5p-770m.json"
ds_path = "data/_06_generated_splits/droidcollection/train.parquet"

Current directory: projects2/thesiscopy


Metrics computation

In [ ]:
def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, and F1 from model logits."""
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = logits.argmax(axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def generate_predictions(model, tokenizer, dataset, model_type, device="cuda"):
    model.to(device)
    model.eval()
    preds = []
    labels = []

    for sample in dataset:
        if model_type == "causal":
            prompt = (
                "Classify the following code (output either HUMAN_GENERATED or MACHINE_GENERATED):\n\n"
                + sample["raw_code"]
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=10)
            text = tokenizer.decode(out[0], skip_special_tokens=True)
            pred_text = text.split("Answer:")[-1].strip().lower()
        else:  # CodeT5
            inputs = tokenizer(
                "classify: " + sample["raw_code"],
                return_tensors="pt"
            ).to(device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=10)
            pred_text = tokenizer.decode(out[0], skip_special_tokens=True).lower()

        pred_text = pred_text.strip().replace(".", "")
        # Fuzzy match because LLMs might add extra text
        if "human" in pred_text:
            pred_label = 0
        elif "machine" in pred_text:
            pred_label = 1
        else:
            pred_label = LABEL_ID.get(pred_text.upper(), -1)
            
        preds.append(pred_label)
        labels.append(sample["labels"])

    return preds, labels


def compute_f1(preds, labels):
    # Filter out -1 if generation failed completely
    valid_idx = [i for i, p in enumerate(preds) if p != -1]
    if not valid_idx:
        return {"accuracy": 0, "precision": 0, "recall": 0, "f1": 0}
        
    v_preds = [preds[i] for i in valid_idx]
    v_labels = [labels[i] for i in valid_idx]
    
    acc = accuracy_score(v_labels, v_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        v_labels, v_preds,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

### Run grid search + CV fold

In [ ]:
BEST_METRIC = "f1"
EARLY_STOP_PATIENCE = 2


def run_cv(dataset, tokenizer, model_name, folds, lr, batch_size, grad_acc_steps, epochs, output_dir, window=512):
    """Run stratified K-fold training and return per-fold metrics."""
    labels = dataset["label"]
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)

    model_name_l = model_name.lower()
    model_type = "encoder"
    if "codet5" in model_name_l:
        model_type = "seq2seq"
    elif "deepseek" in model_name_l:
        model_type = "causal"

    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(
        skf.split(np.arange(len(labels)), labels), start=1
    ):        
        out_dir = Path(output_dir) / f"fold_{fold}"
        os.environ["TENSORBOARD_LOGGING_DIR"] = str(out_dir)

        print(f"\n── Start fold {fold}/{folds} ──")

        train_ds = dataset.select(train_idx)
        test_ds = dataset.select(test_idx)
        print("Train & Test data selected\n")

        print("Tokenizing train data into chunks...")
        train_tok = tokenize_and_chunk(train_ds, tokenizer, window, model_type=model_type)
        print("Tokenizing test data into chunks...")
        test_tok = tokenize_and_chunk(test_ds, tokenizer, window, model_type=model_type)

        print("\nLoading model for fold...")
        model = load_model(model_name, tokenizer)

        train_tok = format_labels(train_tok, tokenizer, model_type, window)
        test_tok = format_labels(test_tok, tokenizer, model_type, window)

        collator = DefaultDataCollator() if model_type == "encoder" else DataCollatorForSeq2Seq(tokenizer, model)

        args = TrainingArguments(
            output_dir=str(out_dir),
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            gradient_accumulation_steps=grad_acc_steps,
            gradient_checkpointing=True if model_type == "causal" else False,
            learning_rate=lr,
            num_train_epochs=epochs,
            evaluation_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            logging_steps=200,
            report_to=["tensorboard"],
            load_best_model_at_end=True,
            metric_for_best_model=BEST_METRIC,
            bf16=True
        )

        trainer = Trainer(
            model=model,
            args=args,
            data_collator=collator,
            train_dataset=train_tok,
            eval_dataset=test_tok,
            compute_metrics=compute_metrics if model_type == "encoder" else None,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)]
        )

        print("\n===========================")
        print("Training Arguments:")
        print(f" Effective batch size: {batch_size * grad_acc_steps}")
        print(f" Learning rate: {lr}")
        print(f" Epochs: {epochs}")
        print("===========================")

        print("\n  Training...")
        trainer.train()

        print("\n  Evaluating...")
        if model_type == "encoder":
            metrics = trainer.evaluate()
        else:
            preds, labels = generate_predictions(
                model,
                tokenizer,
                test_tok,
                model_type
            )
            metrics = compute_f1(preds, labels)

        fold_metrics.append(metrics)
        torch.cuda.empty_cache()

    metric_keys = fold_metrics[0].keys()
    mean_metrics = {k: float(np.mean([m[k] for m in fold_metrics])) for k in metric_keys}
    std_metrics = {k: float(np.std([m[k] for m in fold_metrics])) for k in metric_keys}

    print(f"\n── All folds complete ──")

    return {
        "folds": fold_metrics,
        "mean": mean_metrics,
        "std": std_metrics,
    }

def run_grid_search(config_path, dataset_path):
    """Run a hyperparameter grid search with stratified CV."""
    cfg = json.load(open(config_path))
    model_name = cfg["model_name"]
    print("\n[GRID SEARCH] model_name =", repr(model_name))
    grid = ParameterGrid(cfg["params"])

    results_dir = Path("data/fine_tune/folds") / Path(ds_path).parent.name / Path(model_name)
    results_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = load_tokenizer(model_name)
    dataset = Dataset.from_parquet(dataset_path, columns=["code", "label"])
    
    for run, params in enumerate(grid, start=1):
        lr = params["learning_rate"]
        bs = params["batch_size"]
        gacc = params["grad_acc_step"]
        run_name = f"lr{lr}_bs{bs}"
        out_dir = results_dir / run_name
        out_dir.mkdir(exist_ok=True)

        print("Running grid search")
        print(f"     (run {run}/{len(grid)})")
        print("────────────────────")

        metrics = run_cv(
            dataset,
            tokenizer=tokenizer,
            model_name=model_name,
            folds=cfg["cv_folds"],
            lr=lr,
            batch_size=bs,
            grad_acc_steps=gacc,
            epochs=params["epoch"],
            output_dir=str(out_dir),
            window=cfg.get("window", 512)
        )

        with open(out_dir / "metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)

run_grid_search(config_path=cfg_path, dataset_path=ds_path)


[GRID SEARCH] model_name = 'Salesforce/codet5p-770m'
Running grid search
     (run 1/2)
────────────────────

── Start fold 1/3 ──
Train & Test data selected

Tokenizing train data into chunks...
Tokenizing test data into chunks...

Loading model for fold...

Training Arguments:
 Effective batch size: 64
 Learning rate: 3e-05
 Epochs: 3

  Training...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss


### Train best model

In [ ]:
torch.cuda.empty_cache()

def train_best_model(config_path, dataset_path, metric_name="f1"):
    """Train a final model on the full dataset using best CV settings.

    Args:
        config_path: Path to the JSON config used for the grid search.
        dataset_path: Path to a parquet dataset with "code" and "label" columns.
        metric_name: Metric key used to select the best run (default: "f1").
    """
    cfg = json.load(open(config_path))
    model_name = cfg["model_name"]
    window = cfg.get("window", 512)

    model_name_l = model_name.lower()
    model_type = "encoder"
    if "codet5" in model_name_l:
        model_type = "seq2seq"
    elif "deepseek" in model_name_l:
        model_type = "causal"

    tokenizer = load_tokenizer(model_name)

    results_dir = Path("data/fine_tune/models") / Path(ds_path).parent.name / Path(model_name)
    results_dir.mkdir(parents=True, exist_ok=True)
    
    print("── Retrieving best hyperparameters...")

    best = None
    # Use metric_name which is usually "f1" or "accuracy"
    eval_key = f"eval_{metric_name}" if model_type == "encoder" else metric_name
    
    folds_base_path = Path(f"data/fine_tune/folds/{Path(ds_path).parent.name}/{model_name}")
    for run_dir in sorted(folds_base_path.glob("lr*_bs*")):
        metrics_path = run_dir / "metrics.json"
        if not metrics_path.exists():
            continue
        metrics = json.load(open(metrics_path))
        score = metrics.get("mean", {}).get(eval_key)
        if score is None:
            continue

        if best is None or score > best["score"]:
            lr_str, bs_str = run_dir.name.replace("lr", "").split("_bs")
            best = {
                "score": score,
                "lr": float(lr_str),
                "batch_size": int(bs_str),
            }

    if best is None:
        raise ValueError(f"No metrics found with key '{eval_key}' to select the best run in {folds_base_path}")

    dataset = Dataset.from_parquet(dataset_path, columns=["code", "label"])

    print("\nTokenizing full dataset into chunks...")
    train_tok = tokenize_and_chunk(dataset, tokenizer, window, model_type=model_type)
    train_tok = format_labels(train_tok, tokenizer, model_type, window)

    print("\nLoading model for final training...")
    model = load_model(model_name, tokenizer)

    collator = DefaultDataCollator() if model_type == "encoder" else DataCollatorForSeq2Seq(tokenizer, model)

    args = TrainingArguments(
        output_dir=str(results_dir),
        per_device_train_batch_size=best["batch_size"],
        gradient_accumulation_steps=cfg["params"]["grad_acc_step"][0],
        learning_rate=best["lr"],
        num_train_epochs=cfg["params"]["epoch"][0],
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=100,
        report_to=["tensorboard"],
        bf16=True
    )

    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collator,
        train_dataset=train_tok
    )

    print("\nTraining final model...")
    trainer.train()
    trainer.save_model(str(results_dir))

    with open(results_dir / "best_hyperparams.json", "w") as f:
        json.dump(best, f, indent=2)
    
    print(f"\n── Model saved to {str(results_dir)}")

train_best_model(config_path=cfg_path, dataset_path=ds_path)

── Retrieving best hyperparameters...

Tokenizing full dataset into chunks...

Loading model for final training...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training final model...


Step,Training Loss
100,0.484500
200,0.303648
300,0.259773
400,0.250108
500,0.227656
600,0.198532
700,0.185085
800,0.189213
900,0.164224
1000,0.157639


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


── Model saved to data/fine_tune/models/droidcollection/microsoft/codebert-base
